In [1]:
import pandas as pd
from helpers import get_factor, get_price

In [2]:
pd.options.mode.chained_assignment = None

In [3]:
CDF = pd.read_csv("../production/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [4]:
smard = pd.read_csv("Gro_handelspreise_202301010000_202401010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [5]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [6]:
CDF2 = CDF.loc[CDF.produced_at > "2022-12-31 23:50"].loc[CDF.produced_at < "2024-01-01 00:00"]

In [7]:
len(CDF2.groupby('produced_at').sum()) * 4

35036

In [8]:
dataset = CDF2.merge(seem, left_on="variable", right_on="sseid")

In [9]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [10]:
magic2 = magic[["value"]]

In [11]:
#magic2.sort_values(["produced_at", "value"])

In [12]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [13]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")

In [14]:
merged["revenue"] = merged["value"] * merged["price"]
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [15]:
tmp1.reset_index(inplace=True)

In [16]:
revenue = tmp1[["plantid", "revenue"]]

In [17]:
tmp1

,plantid,value,price,revenue
0,06-02-B10117A007,707521,833736.96,6.933753e+07
1,BB23020490,1182664,833736.96,1.124468e+08
2,BB45025564,11629654,833736.96,1.185297e+09
3,BB45025611,8443056,833736.96,8.770335e+08
4,BE166928,1284475,833736.96,1.283071e+08
5,BE169709,347644,833736.96,3.511908e+07
6,BE172654,1375621,833736.96,1.436704e+08
7,BE172656,1135173,833736.96,1.200585e+08
8,BWpf-450-1020129-00000000,306853,833736.96,4.400377e+07
9,BWpf-450-1195689-00000000,199574,833736.96,1.950829e+07


In [18]:
#dataset.sort_values(["plantid", "produced_at"])

In [19]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [20]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [21]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [22]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [23]:
co2s2

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
52,2023,BB16018798,CO2,Air,2.850000e+08,9,Mio. t,0.285,CO2 [Mio. t]
84,2023,BB23020389,CO2,Air,4.340000e+08,9,Mio. t,0.434,CO2 [Mio. t]
163,2023,BB23020490,CO2,Air,3.015000e+09,9,Mio. t,3.015,CO2 [Mio. t]
369,2023,BB23022811,CO2,Air,1.500000e+08,9,Mio. t,0.150,CO2 [Mio. t]
411,2023,BB45025564,CO2,Air,1.413400e+10,9,Mio. t,14.134,CO2 [Mio. t]
...,...,...,...,...,...,...,...,...,...
15336,2023,ST18046,CO2,Air,2.210000e+08,9,Mio. t,0.221,CO2 [Mio. t]
15459,2023,TH30013152,CO2,Air,2.620000e+08,9,Mio. t,0.262,CO2 [Mio. t]
15490,2023,TH62013494,CO2,Air,1.290000e+08,9,Mio. t,0.129,CO2 [Mio. t]
15566,2023,TH72012874,CO2,Air,1.770000e+08,9,Mio. t,0.177,CO2 [Mio. t]


In [24]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [25]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [26]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [27]:
tmp2 = tmp1

In [28]:
coal_cost_per_t = 103.5 or 120
co2_cost = 70
#electricity_price = 78.50

In [29]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [30]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [31]:
profit = tmp2[["plantid", "revenue", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

In [32]:
profit.to_csv("profit.csv", index=False)

In [33]:
profit.sort_values("profit")

,plantid,revenue,profit
0,BB23020490,112.446805,-196.918413
12,BWpf-450-2948214-00000000,247.873328,-139.366872
35,NW500-0342658,118.681204,-106.696332
14,BYS00041,113.533971,-88.499362
20,MV30000226,135.982097,-61.338277
9,BWpf-450-1741292-00000000,56.747003,-45.816059
22,NI01241117210,118.842134,-45.172941
3,BE166928,128.307113,-44.774484
17,BYS00114,16.650636,-42.992697
8,BWpf-450-1195689-00000000,19.508291,-31.911410
